## Gradient descent - linear regression model

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jit
from functools import partial
import matplotlib.pyplot as plt

Generate synthetic data, with noise

In [ ]:
key = jax.random.PRNGKey(0)
key_x, key_noise = jax.random.split(key)

n_samples = 100 # number of data points
true_w = 3.5
true_b = -1.0

x = jax.random.normal(key_x, (n_samples,))
noise = 0.1 * jax.random.normal(key_noise, (n_samples,))
y = true_w * x + true_b + noise

Define the linear model (that we want to fit to the synthetic data)

In [ ]:
def model(params, x):
    """
    Linear model: y = w * x + b
    params = (w, b)
    """
    w, b = params
    return w * x + b

Define the loss function (mean squared error loss between data and model output), and its gradient

In [ ]:
def loss_fn(params, x, y):
    """
    Mean Squared Error loss
    """
    y_pred = model(params, x)
    return jnp.mean((y_pred - y) ** 2)

loss_grad_fn = grad(loss_fn) #uses jax grad

Define a single gradient descent step

In [ ]:
@jit
def gradient_descent_step(params, x, y, learning_rate):
    grads = loss_grad_fn(params, x, y)
    new_params = tuple(
        param - learning_rate * grad
        for param, grad in zip(params, grads)
    )
    return new_params

Make training loop using the gradient descent step

In [ ]:
def train(params, x, y, learning_rate=0.1, num_epochs=1000, info_every=100):
    for epoch in range(num_epochs):
        params = gradient_descent_step(params, x, y, learning_rate)

        # print info every 'info_every' epochs
        if epoch % info_every == 0:
            current_loss = loss_fn(params, x, y)
            print(
                f"Epoch {epoch:4d} | "
                f"Loss = {current_loss:.6f} | "
                f"w = {params[0]:.4f}, b = {params[1]:.4f}"
            )

    return params

Run the training

In [ ]:
initial_params = (jnp.array(0.0), jnp.array(0.0))

final_params = train(initial_params, x, y, learning_rate=0.1, num_epochs=1000, info_every=50)

print("\nFinal parameters:")
print(f"w = {final_params[0]:.4f}")
print(f"b = {final_params[1]:.4f}")

Plot the true line, synthetic data, and model line

In [ ]:
# Create scatter plot of synthetic data
plt.figure(figsize=(10, 6))
plt.scatter(x, y, alpha=0.6, label='Synthetic data')

# Plot true line
x_line = jnp.linspace(x.min(), x.max(), 100)
y_true_line = true_w * x_line + true_b
plt.plot(x_line, y_true_line, 'k-', linewidth=1, label=f'True line (w={true_w}, b={true_b})')

# Plot model line
y_pred_line = final_params[0] * x_line + final_params[1]
plt.plot(x_line, y_pred_line, 'r-', linewidth=1, label=f'Fitted line (w={final_params[0]:.4f}, b={final_params[1]:.4f})')

plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()